In [2]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib as mpl
import pandas as pd
from matplotlib import animation

from tools.sportec_data import SportecData

pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 30)

%matplotlib inline
mpl.rcParams['animation.embed_limit'] = 100

In [3]:
# It can take about one or two minutes to parse tracking data using kloppy
match_id = "J03WMX"
match = SportecData(match_id)
match.lineup.head()

Loading the tracking data...
Transforming the tracking data coordinates...


,team_id,team_name,home_away,player_id,uniform_number,object_id,player_name,starting,playing_position,captain
0,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0027KL,2,away_2,Dayot Upamecano,True,RCB,False
1,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-J017RE,4,away_4,M. de Ligt,True,LCB,False
2,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0027G0,5,away_5,Benjamin Pavard,True,RB,False
3,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0002F5,6,away_6,Joshua Kimmich,True,RDM,False
4,DFL-CLU-00000G,FC Bayern München,away,DFL-OBJ-0027G6,7,away_7,Serge Gnabry,True,CF,False


In [4]:
match.tracking.iloc[70705:70710]

,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,away_27_x,away_27_y,away_27_d,away_27_s,away_25_x,away_25_y,...,home_3_y,home_3_d,home_3_s,home_25_x,home_25_y,home_25_d,home_25_s,away_39_x,away_39_y,away_39_d,away_39_s,away_42_x,away_42_y,away_42_d,away_42_s
70705,1,2828.20,80705,dead,DFL-CLU-000008,68.36,0.82,0.0,NaN,106.07,34.21,None,5.38,47.41,29.18,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
70706,1,2828.24,80706,dead,DFL-CLU-000008,68.36,0.82,0.0,NaN,106.02,34.26,None,5.76,47.42,29.12,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
70707,1,2828.28,80707,dead,DFL-CLU-000008,68.36,0.82,0.0,NaN,105.98,34.31,None,6.09,47.44,29.06,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
70708,2,0.00,100000,alive,DFL-CLU-000008,52.57,33.54,0.0,0.51,98.22,33.97,None,0.00,52.94,42.53,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN
70709,2,0.04,100001,alive,DFL-CLU-000008,51.87,33.50,0.0,0.00,98.15,33.97,None,6.34,52.87,42.47,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,NaN


In [5]:
from autoevent.poss import PossessionDetector

In [6]:
detector = PossessionDetector(match.tracking)

In [7]:
result = detector.run()

In [8]:
result.head()

,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed,away_27_x,away_27_y,away_27_d,away_27_s,away_25_x,away_25_y,...,dist_home_8,ball_control,controller_id,controller_team,controller_distance,duel_players,control_sequence_id,control_sequence_type,control_sequence_player,is_loss,loss_player,loss_team,is_gain,gain_player,gain_team
0,1,0.00,10000,alive,DFL-CLU-00000G,52.321429,34.169286,0.00,NaN,99.23,33.82,None,0.00,54.03,21.32,...,NaN,possession,away_7,away,0.706502,<NA>,1,possession,away_7,False,<NA>,<NA>,False,<NA>,<NA>
1,1,0.04,10001,alive,DFL-CLU-00000G,52.896429,34.190714,0.00,NaN,99.21,33.82,None,2.21,53.95,21.32,...,NaN,possession,away_7,away,1.299983,<NA>,1,possession,away_7,False,<NA>,<NA>,False,<NA>,<NA>
2,1,0.08,10002,alive,DFL-CLU-00000G,53.470714,34.212857,0.01,NaN,99.18,33.82,None,2.35,53.77,21.35,...,NaN,possession,away_7,away,1.947579,<NA>,1,possession,away_7,True,away_7,away,False,<NA>,<NA>
3,1,0.12,10003,alive,DFL-CLU-00000G,54.044286,34.235714,0.01,NaN,99.15,33.82,None,2.49,53.57,21.36,...,NaN,no_possession,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,False,<NA>,<NA>,False,<NA>,<NA>
4,1,0.16,10004,alive,DFL-CLU-00000G,54.616667,34.260000,0.02,NaN,99.12,33.82,None,2.63,53.38,21.39,...,NaN,no_possession,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,False,<NA>,<NA>,False,<NA>,<NA>


In [9]:
import importlib
import tools.config as cfg
import autoevent.setpiece_trigger as trg
import autoevent.set as st

importlib.reload(cfg)
importlib.reload(trg)
importlib.reload(st)

<module 'autoevent.set' from '/Users/kimseungwoo/Desktop/시립대/Automatic Event Detection/autoevent/set.py'>

In [10]:
from autoevent.set import SetPieceDetector, SetPieceConfig
import inspect, autoevent.set as set_mod

print("loaded from:", inspect.getsourcefile(set_mod.SetPieceDetector))

detector = SetPieceDetector(result)  # result는 poss 이후 tracking
print("instance tol:", detector.cfg.kickoff_x_tol, detector.cfg.half_tol)
test = SetPieceDetector(result).run()

loaded from: /Users/kimseungwoo/Desktop/시립대/Automatic Event Detection/autoevent/set.py
instance tol: 2.0 5.0


In [11]:
kickoff_test = test[test['set_piece_type'] == 'kickoff']
kickoff_test[["period_id", "frame_id", "timestamp","ball_state", "set_piece_type", "trigger_player"]]

,period_id,frame_id,timestamp,ball_state,set_piece_type,trigger_player
0,1,10000,0.00,alive,kickoff,away_7
13016,1,23016,520.64,alive,kickoff,home_11
70708,2,100000,0.00,alive,kickoff,home_11
125641,2,154933,2197.32,alive,kickoff,away_8
137642,2,166934,2677.36,alive,kickoff,home_14


In [12]:
from autoevent.set import SetPieceDetector, SetPieceConfig
detector = SetPieceDetector(result)
test = SetPieceDetector(result).run()

In [13]:
penalty_test = test[test['set_piece_type'] == 'penalty_kick']
penalty_test[["period_id", "frame_id", "timestamp","ball_state", "set_piece_type", "trigger_player"]]

,period_id,frame_id,timestamp,ball_state,set_piece_type,trigger_player
123308,2,152600,2104.0,alive,penalty_kick,home_7


In [21]:
test[123307:123311][
    ["frame_id", "timestamp", "ball_state", "set_piece_type", 
    "trigger_player", "controller_id", "dist_home_7",
]]

,frame_id,timestamp,ball_state,set_piece_type,trigger_player,controller_id,dist_home_7
123307,152599,2103.96,dead,<NA>,<NA>,<NA>,NaN
123308,152600,2104.00,alive,penalty_kick,home_7,home_7,0.984338
123309,152601,2104.04,alive,<NA>,<NA>,home_7,1.911380
123310,152602,2104.08,alive,<NA>,<NA>,<NA>,2.910501


In [15]:
importlib.reload(cfg)
importlib.reload(trg)
importlib.reload(st)

from autoevent.set import SetPieceDetector
detector = SetPieceDetector(result)
test = SetPieceDetector(result).run()

In [16]:
corner_kick_test = test[test['set_piece_type'] == 'corner_kick']
corner_kick_test[[
    "period_id", "frame_id", "timestamp", "ball_state", 
    "set_piece_type", "trigger_player",
    "dist_home_11", "dist_away_6", "dist_home_8",

    ]]

,period_id,frame_id,timestamp,ball_state,set_piece_type,trigger_player,dist_home_11,dist_away_6,dist_home_8
22625,1,32625,905.00,alive,corner_kick,home_11,0.752379,39.566348,NaN
25923,1,35923,1036.92,alive,corner_kick,away_6,29.242425,1.087396,NaN
41184,1,51184,1647.36,alive,corner_kick,home_11,0.897252,34.228556,NaN
81759,2,111051,442.04,alive,corner_kick,home_11,0.348746,38.764889,NaN
92244,2,121536,861.44,alive,corner_kick,home_11,0.287894,39.919743,NaN
92900,2,122192,887.68,alive,corner_kick,home_11,2.295027,35.969912,NaN
105687,2,134979,1399.16,alive,corner_kick,away_6,NaN,1.045346,41.584445
108845,2,138137,1525.48,alive,corner_kick,away_6,NaN,0.534085,37.015212
113589,2,142881,1715.24,alive,corner_kick,home_8,NaN,27.736505,0.295083
120178,2,149470,1978.80,alive,corner_kick,home_8,NaN,26.862168,1.988565


코너킥 키커의 공과의 거리가 최대 2.29m인 경우 때문에 Ball in PZ를 못 잡고 누락되는 경우가 있음.

In [17]:
# 공과의 거리가 1m가 넘음
test.iloc[25922:25925][[
    'ball_state', 'set_piece_type', 'controller_id', 'trigger_player', "dist_away_6",
    'ball_x','ball_y', 'away_6_x', 'away_6_y'
    ]]

,ball_state,set_piece_type,controller_id,trigger_player,dist_away_6,ball_x,ball_y,away_6_x,away_6_y
25922,dead,<NA>,<NA>,<NA>,NaN,NaN,NaN,-0.55,0.35
25923,alive,corner_kick,away_6,away_6,1.087396,0.378095,1.094762,-0.45,0.39
25924,alive,<NA>,away_6,<NA>,2.104172,0.696429,2.283571,-0.42,0.50


In [ ]:
# 공과의 거리가 2.3m
test.iloc[92899:92902][[
    'timestamp', "dist_home_11",
    'controller_id', 'trigger_player',
    'ball_x','ball_y', 'home_11_x', 'home_11_y'
    ]]

,timestamp,dist_home_11,controller_id,trigger_player,ball_x,ball_y,home_11_x,home_11_y
92899,887.64,NaN,<NA>,<NA>,NaN,NaN,103.28,-0.08
92900,887.68,2.295027,<NA>,home_11,104.479524,1.982143,103.34,-0.01
92901,887.72,3.166511,<NA>,<NA>,104.428571,3.047857,103.38,0.06


In [191]:
events = match.format_events_for_syncer().rename(columns={"utc_timestamp": "timestamp"})

In [192]:
events[events['spadl_type'].str.startswith('corner_')]

,period_id,timestamp,player_id,spadl_type,start_x,start_y,success
271,1,2023-05-27 13:45:16.620,home_11,corner_crossed,105.0,68.0,False
308,1,2023-05-27 13:47:29.527,away_6,corner_crossed,0.0,0.0,False
443,1,2023-05-27 13:57:39.106,home_11,corner_crossed,105.0,68.0,False
814,2,2023-05-27 14:43:05.068,home_11,corner_crossed,105.0,0.0,False
912,2,2023-05-27 14:50:03.681,home_11,corner_crossed,105.0,0.0,True
915,2,2023-05-27 14:50:29.554,home_11,corner_crossed,105.0,0.0,False
1020,2,2023-05-27 14:59:01.219,away_6,corner_crossed,0.0,68.0,True
1041,2,2023-05-27 15:01:10.096,away_6,corner_crossed,0.0,0.0,False
1079,2,2023-05-27 15:04:17.377,home_8,corner_crossed,105.0,0.0,True
1136,2,2023-05-27 15:08:43.547,home_8,corner_short,105.0,68.0,True
